# 4 Data Exploration

This notebook is a Renku-friendly demo for exploring the published MOTEL data product.

It stays focused on operations and usage:
- the notebook shows the user workflow
- `config.py` stores paths and default demo parameters
- `exploration_utils.py` stores reusable data loading and analysis helpers


## Run Order

Run the notebook from top to bottom.

For a Renku demo, a good flow is:
1. load and inspect the prepared tables
2. run one use case with the default parameters
3. change one parameter live and rerun the relevant section

Instruction:
- Run every cell in sequence the first time.
- After that, you usually only need to rerun one input cell and the result cell directly below it.


## Step 1: Load The Data And Helper Functions

What this step does:
- imports plotting and table display tools
- imports project defaults from `config.py`
- imports reusable exploration logic from `exploration_utils.py`
- loads the canonical `linked_entity` dataset and converts it into analysis-ready tables

Instruction:
- Run the next cell once.
- If it fails, check that the notebook is being run from the `4_data_explore` folder and that the project files are present.
- The output should be a single number showing how many linked entities were loaded.


In [ ]:
import pandas as pd
import plotly.express as px
from IPython.display import display

from config import (
    DEFAULT_ATTRIBUTE_OF_INTEREST,
    DEFAULT_CARRIER_QUERY,
    DEFAULT_SOURCE_KEYWORDS,
    DEFAULT_TECHNOLOGY_KEYWORDS,
    LINKED_ENTITY_PATH,
    MAPPING_DIR,
    VOCAB_DIR,
)
from exploration_utils import (
    build_analysis_tables,
    build_overview_df,
    filter_carriers_by_query,
    filter_entities_by_keywords,
    filter_source_attributes_by_keywords,
    load_lookup_tables,
    read_yaml,
    select_attribute_values,
    summarize_attributes_for_entities,
    summarize_source_attribute_coverage,
    summarize_sources_for_entities,
    summarize_technologies_by_carrier,
    top_attribute_types_by_source,
)

# Load the canonical linked-entity data.
linked_entities = read_yaml(LINKED_ENTITY_PATH)

# Load lookup tables so IDs can be shown with readable labels.
lookups = load_lookup_tables(MAPPING_DIR, VOCAB_DIR)

# Build flat tables that are easier to inspect and plot in the notebook.
tables = build_analysis_tables(linked_entities, lookups)

entities_df = tables["entities_df"]
values_df = tables["values_df"]
sources_df = tables["sources_df"]
source_attributes_df = tables["source_attributes_df"]
carriers_df = tables["carriers_df"]

len(linked_entities)

## Step 2: Inspect The Loaded Tables

What this step does:
- gives a compact summary of the dataset
- shows a preview of the main entity table
- helps you confirm that the load step worked before moving into the use cases

Instruction:
- Run the next cell.
- Read the overview table first.
- Then scan the entity preview to see the columns that will be reused later.
- If the preview looks wrong, stop here and debug before continuing.


In [ ]:
# Build a high-level summary for a quick dataset health check.
overview_df = build_overview_df(entities_df, values_df, sources_df, carriers_df)
display(overview_df)

# Preview the main exploration table used in the use cases below.
entities_df[
    [
        "linked_entity_id",
        "tech_id",
        "tech_name",
        "process_id",
        "process_name",
        "reference_year",
        "input_carriers",
        "output_carriers",
        "source_count",
        "value_count",
    ]
].head(10)

## How To Use The Demo Inputs

The next sections expose only a few parameters.

Instruction:
- Edit values only in the small input cells.
- Leave the result cells unchanged unless you are extending the workflow.
- When you change an input, rerun that input cell and then the result cell below it.

This keeps the notebook easy to operate live while the heavier logic stays in reusable Python files.

## Use Case 1: Search By Technology Or Process

Typical user question:

> Which hydrogen or fuel-cell related technologies are available, which sources support them, and how do their capital costs compare across years?

This use case is good for a demo when the audience starts from a technology family rather than from a source or a carrier.

### Step 3A: Choose Technology Keywords And One Attribute

Instruction:
- Edit `technology_keywords` if you want a different topic, for example `['solar']`, `['battery']`, or `['methanol']`.
- Edit `attribute_of_interest` if you want a different numeric comparison, for example `Technical Efficiency` or `Economic Lifetime`.
- Run the next cell after changing either input.


In [ ]:
# Set the technology or process terms you want to search for.
technology_keywords = DEFAULT_TECHNOLOGY_KEYWORDS

# Choose the attribute that should be compared across the selected technologies.
attribute_of_interest = DEFAULT_ATTRIBUTE_OF_INTEREST

display(pd.DataFrame({"technology_keywords": technology_keywords}))
attribute_of_interest

### Step 3B: Run The Technology Search And Read The Outputs

What this step does:
- filters the entity table using your keywords
- shows the matching entities first
- summarizes which sources support those entities
- summarizes which attributes appear most often
- plots the selected numeric attribute across years

Instruction:
- Run the next cell.
- Read the outputs from top to bottom.
- For a live demo, start with the table, then the source bar chart, then the final trend line.


In [ ]:
# Filter the entity table using the chosen keywords.
selected_entities = filter_entities_by_keywords(entities_df, technology_keywords)
display(
    selected_entities[
        [
            "linked_entity_id",
            "tech_id",
            "tech_name",
            "process_id",
            "process_name",
            "reference_year",
            "input_carriers",
            "output_carriers",
            "source_count",
            "value_count",
        ]
    ]
)

# Summarize how strongly each source contributes to the filtered set.
selected_source_summary = summarize_sources_for_entities(
    sources_df,
    selected_entities["linked_entity_id"],
)

# Summarize which attributes are most common across the filtered entities.
selected_attribute_summary = summarize_attributes_for_entities(
    values_df,
    selected_entities["linked_entity_id"],
)

# Keep only the numeric records for the selected attribute so they can be plotted.
selected_attribute_values = select_attribute_values(
    values_df,
    selected_entities["linked_entity_id"],
    attribute_of_interest,
)

px.bar(
    selected_source_summary,
    x="source_name",
    y="linked_attributes",
    color="linked_entities",
    title="Source coverage for the selected technologies",
    labels={"source_name": "Source", "linked_attributes": "Linked attributes"},
).show()

px.bar(
    selected_attribute_summary,
    x="records",
    y="attribute_name",
    orientation="h",
    title="Most common attributes across the selected technologies",
).show()

# Show the exact rows behind the final comparison chart.
display(
    selected_attribute_values[
        [
            "linked_entity_id",
            "tech_name",
            "process_name",
            "reference_year",
            "attribute_name",
            "value",
            "time_index",
        ]
    ]
)

px.line(
    selected_attribute_values,
    x="reference_year",
    y="value_numeric",
    color="tech_name",
    markers=True,
    hover_data=["process_name", "linked_entity_id", "time_index"],
    title=f"{attribute_of_interest} by reference year",
    labels={"reference_year": "Reference year", "value_numeric": attribute_of_interest},
).show()

## Use Case 2: Search By Source

Typical user question:

> Which sources contribute the most linked attributes, what kinds of attributes do they contribute, and how wide is their technology coverage?

This use case is helpful when users want to understand provenance and dataset coverage.

### Step 4A: Choose One Or More Source Keywords

Instruction:
- Edit `source_keywords` to focus on one source family or compare several.
- Good demo examples are short fragments such as `['VSE']`, `['ShareRef']`, or `['DanishEnergyAgency']`.
- Run the next cell after updating the list.


In [ ]:
# Set the source names or source-name fragments you want to inspect.
source_keywords = DEFAULT_SOURCE_KEYWORDS
display(pd.DataFrame({"source_keywords": source_keywords}))

### Step 4B: Run The Source Analysis

What this step does:
- filters the source-to-attribute table by your chosen source keywords
- summarizes how many linked attributes each source contributes
- shows how broad each source is across technologies and processes
- shows the most common attribute types contributed by each source

Instruction:
- Run the next cell.
- Use the summary table to orient yourself first.
- Then use the three charts to explain contribution volume, breadth, and attribute mix.


In [ ]:
# Keep only the source-attribute records that match the requested source keywords.
selected_source_attributes = filter_source_attributes_by_keywords(
    source_attributes_df,
    source_keywords,
)

# Build summary views used in the table and charts below.
source_attribute_summary = summarize_source_attribute_coverage(selected_source_attributes)
top_attribute_types = top_attribute_types_by_source(selected_source_attributes)

display(source_attribute_summary)

px.bar(
    source_attribute_summary,
    x="source_name",
    y="linked_attributes",
    color="distinct_attribute_types",
    title="How many linked attributes each source contributes",
    labels={"source_name": "Source", "linked_attributes": "Linked attributes"},
).show()

px.bar(
    source_attribute_summary,
    x="source_name",
    y="distinct_technologies",
    color="distinct_processes",
    title="Technology coverage breadth by source",
    labels={"source_name": "Source", "distinct_technologies": "Distinct technologies"},
).show()

px.bar(
    top_attribute_types,
    x="records",
    y="attribute_name",
    color="source_name",
    facet_col="source_name",
    facet_col_wrap=2,
    orientation="h",
    title="Top attribute types proposed by each selected source",
).show()

## Use Case 3: Search By Energy Carrier

Typical user question:

> If I start from a carrier such as hydrogen, which technologies consume it or produce it, and in which years do those entities appear?

This use case is useful when the audience thinks in terms of inputs and outputs rather than technologies.

### Step 5A: Choose A Carrier Query

Instruction:
- Edit `carrier_query` to a carrier name or fragment such as `hydrogen`, `electricity`, `methanol`, or `ammonia`.
- Run the next cell after changing the query.


In [ ]:
# Set the carrier name or partial carrier name you want to explore.
carrier_query = DEFAULT_CARRIER_QUERY
carrier_query

### Step 5B: Run The Carrier Analysis

What this step does:
- filters the carrier table using your carrier query
- shows all matching carrier rows first
- groups them into a simpler technology-level summary
- plots how many linked entities are connected to the carrier on the input or output side

Instruction:
- Run the next cell.
- Use the first table to show the raw evidence.
- Use the second table and chart to present the simplified story.


In [ ]:
# Keep only the carrier rows that match the requested carrier query.
matched_carriers = filter_carriers_by_query(carriers_df, carrier_query)

# Build a smaller summary table for presentation.
technology_by_carrier = summarize_technologies_by_carrier(matched_carriers)

display(
    matched_carriers[
        [
            "linked_entity_id",
            "tech_id",
            "tech_name",
            "process_id",
            "process_name",
            "reference_year",
            "direction",
            "carrier_id",
            "carrier_name",
            "carrier_type",
            "share",
            "unit",
        ]
    ].sort_values(["direction", "tech_name", "reference_year"])
)

display(technology_by_carrier)

px.bar(
    technology_by_carrier,
    x="tech_name",
    y="linked_entities",
    color="direction",
    hover_data=["process_name", "first_year", "last_year"],
    title=f"Technologies connected to carrier query: {carrier_query}",
    labels={"tech_name": "Technology", "linked_entities": "Linked entities"},
).show()

## Ad Hoc Query: List Technologies For `Syngas FT C8`

Instruction:
- Run the next cell when you want a quick lookup for `Syngas FT C8`.
- The cell first checks whether `Syngas FT C8` exists as a process name.
- If no process match is found, it falls back to checking whether it exists as a technology name.


In [ ]:
query_name = "Syngas FT C8"

# First try the exact process name requested by the user.
process_matches = entities_df.loc[
    entities_df["process_name"].eq(query_name),
    ["tech_id", "tech_name", "process_id", "process_name", "reference_year"],
].drop_duplicates().sort_values(["tech_name", "reference_year"])

if not process_matches.empty:
    print(f"Technologies linked to process '{query_name}':")
    display(process_matches)
else:
    print(f"No rows found where process_name == '{query_name}'.")
    print("Showing rows where the same value appears as a technology name instead.")
    technology_matches = entities_df.loc[
        entities_df["tech_name"].eq(query_name),
        ["tech_id", "tech_name", "process_id", "process_name", "reference_year"],
    ].drop_duplicates().sort_values(["tech_name", "reference_year"])
    display(technology_matches)

## Next Step

If you want, the next clean upgrade is to add `ipywidgets` on top of this structure.

That would keep the same helper modules and only replace the small input cells with interactive controls.